# YOLO11n N54 R1 — resume smoke to epoch 2

Train/validation only. Requires NVIDIA L4. Never opens the sealed test.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

TRACK = 'n54'
MODE = 'resume'
STOP_AFTER = 2
REPO_URL = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
SOURCE_COMMIT = '45b7b1fab606032fcfb2956d0207171ae8b4c7f6'
SOURCE_QA_SHA256 = 'bcf7470d931d23445785c1103e3c919aca4c696b22738079756572892ce4f632'
WEIGHT_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt'
WEIGHT_SHA256 = '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1'
DRIVE_ROOT = Path('/content/drive/MyDrive/CPV301/Traffic_Sign_Detector')
STATE = DRIVE_ROOT / 'M1_D1_Colab/runs/yolo11n_n54_seed_20261018_r1'
REPO = Path('/content/Traffic_Sign_Detector')
assert MODE in {'fresh', 'resume'}
assert (MODE, STOP_AFTER) in {('fresh', 1), ('resume', 2)}

N54_ROOT = DRIVE_ROOT / 'M1_D1_Colab'
DRIVE_BUNDLE = N54_ROOT / 'm1_colab_train_val_backend_r1.tar.gz'
SIDECAR = N54_ROOT / 'm1_colab_train_val_backend_r1.tar.gz.manifest.json'
BUNDLE_SHA256 = '7ea344cb1e2c166786d0552f215993969efec9b5f06a9bf49310d4390243d641'
BUNDLE_BYTES = 766226800
SIDECAR_SHA256 = '7e0dc5b0144faa5f0be407224eaf8070dd51199fa8a1d2f164b248fd6a06d5b2'
SIDECAR_BYTES = 1499866
PART_SPECS = (('m1_colab_train_val_backend_r1.tar.gz.part-00', 99614720, '53f30f08406fbef5b8451610b2fd44a1f2be42ca6d50f612b68e5ace285e66ff'), ('m1_colab_train_val_backend_r1.tar.gz.part-01', 99614720, '8451fe932760179d4b2e1a9cbdd21fd58744bc6b8f3d2a90e431cece7f546544'), ('m1_colab_train_val_backend_r1.tar.gz.part-02', 99614720, '867c935bf152ddfc3e7919547dccf3682291326b60b956a8100fc1a59b60e98d'), ('m1_colab_train_val_backend_r1.tar.gz.part-03', 99614720, '046fc8ae5d4e0a91196ccd6a6afa422fb1254baea7ae30bc3cab39602b534907'), ('m1_colab_train_val_backend_r1.tar.gz.part-04', 99614720, '6d3751d858232fada5d57e8e892f3e272cd5b9e34efb184971b3f4786842665d'), ('m1_colab_train_val_backend_r1.tar.gz.part-05', 99614720, '938106e4d34743e9b79109feb98d573f317afad44289f792f070f22e3924159b'), ('m1_colab_train_val_backend_r1.tar.gz.part-06', 99614720, '3497c1a10314c74d4b81ba9a5a49653b16ba2e2d52b424fd7655623547cb6408'), ('m1_colab_train_val_backend_r1.tar.gz.part-07', 68923760, '5f5ca85bc392310b2b5693ad9d0723f046eca9fd7980b45c944f22a3b3bc0eea'))
SOURCE_QA = REPO / 'docs/review/yolo11n_dual_track_r1_source_independent_qa_record.md'


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys

def file_hash(path):
    value = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert gpu_name == 'NVIDIA L4', gpu_name
assert len(SOURCE_COMMIT) == 40
assert len(SOURCE_QA_SHA256) == 64
assert len(WEIGHT_SHA256) == 64
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO, check=True)
observed = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
assert observed == SOURCE_COMMIT
assert not subprocess.run(
    ['git', 'status', '--porcelain'], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.11.14'], check=True)
print({'gpu': gpu_name, 'checkout': observed, 'track': TRACK, 'mode': MODE})


In [ ]:
import os
import urllib.request

assert SIDECAR.is_file() and not SIDECAR.is_symlink()
assert SIDECAR.stat().st_size == SIDECAR_BYTES
assert file_hash(SIDECAR) == SIDECAR_SHA256
if DRIVE_BUNDLE.is_file():
    assert not DRIVE_BUNDLE.is_symlink()
    BUNDLE = DRIVE_BUNDLE
    bundle_source = 'single_archive'
else:
    parts = []
    for name, expected_bytes, expected_hash in PART_SPECS:
        part = N54_ROOT / name
        assert part.is_file() and not part.is_symlink()
        assert part.stat().st_size == expected_bytes
        assert file_hash(part) == expected_hash
        parts.append(part)
    BUNDLE = Path('/content/m1_colab_train_val_backend_r1.tar.gz')
    if BUNDLE.exists():
        assert BUNDLE.is_file() and not BUNDLE.is_symlink()
    else:
        with BUNDLE.open('xb') as destination:
            for part in parts:
                with part.open('rb') as source:
                    for chunk in iter(lambda: source.read(1024 * 1024), b''):
                        destination.write(chunk)
    bundle_source = 'verified_drive_parts'
assert BUNDLE.stat().st_size == BUNDLE_BYTES
assert file_hash(BUNDLE) == BUNDLE_SHA256
assert SOURCE_QA.is_file() and file_hash(SOURCE_QA) == SOURCE_QA_SHA256
if MODE == 'fresh':
    assert not STATE.exists(), STATE
else:
    assert (STATE / 'recovery/latest.json').is_file()
weight = REPO / 'models/pretrained/yolo11n.pt'
weight.parent.mkdir(parents=True, exist_ok=True)
if weight.exists():
    assert weight.is_file() and not weight.is_symlink()
    assert file_hash(weight) == WEIGHT_SHA256
else:
    partial = weight.with_suffix('.partial')
    assert not partial.exists()
    urllib.request.urlretrieve(WEIGHT_URL, partial)
    assert file_hash(partial) == WEIGHT_SHA256
    os.replace(partial, weight)
assert file_hash(weight) == WEIGHT_SHA256
print({
    'bundle_source': bundle_source,
    'bundle_sha256': BUNDLE_SHA256,
    'weight_sha256': WEIGHT_SHA256,
})


In [ ]:
command = [
    sys.executable, str(REPO / 'detector/colab/run_yolo11n_n54_r1.py'),
    '--repo-root', str(REPO),
    '--bundle', str(BUNDLE),
    '--bundle-sha256', BUNDLE_SHA256,
    '--state-dir', str(STATE),
    '--mode', MODE,
    '--expected-checkout', SOURCE_COMMIT,
    '--source-qa-record', str(SOURCE_QA),
    '--source-qa-sha256', SOURCE_QA_SHA256,
    '--stop-after-epoch', str(STOP_AFTER),
    '--seed', '20261018', '--epochs', '100', '--imgsz', '1280',
    '--batch', '4', '--workers', '4', '--fraction', '1.0',
    '--patience', '30', '--device', '0', '--tag', 'n54_r1', '--cos-lr',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
execution = json.loads((STATE / 'colab_execution.json').read_text())
latest = json.loads((STATE / 'recovery/latest.json').read_text())
attempts_path = STATE / 'n54_execution_attempts.jsonl'
attempts = [json.loads(line) for line in attempts_path.read_text().splitlines()]
assert execution['status'] == 'PAUSED_FOR_RESUME'
assert execution['sealed_test_accessed'] is False
assert latest['generation'] == f'epoch-{STOP_AFTER:04d}'
assert attempts[-1]['completed_epoch'] == STOP_AFTER
assert attempts[-1]['sealed_test_accessed'] is False
if MODE == 'fresh':
    assert len(attempts) == 1 and attempts[0]['leg'] == 'fresh_epoch_1'
else:
    assert len(attempts) == 2 and attempts[1]['leg'] == 'resume_epoch_2'
    assert attempts[0]['runtime_digest'] != attempts[1]['runtime_digest']
print({'status': 'SMOKE_PASS', 'track': TRACK, 'epoch': STOP_AFTER, 'seal': False})
